In [ ]:
# ==========================================
# 10 - MODEL EVALUATION V1
# KHỞI TẠO MÔI TRƯỜNG
# ==========================================

import sys
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT_DIR))

import src.imports as lib

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
    confusion_matrix
)

RANDOM_SEED = 42
N_SPLITS = 5

id_col = "ma_hoc_sinh"

choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

split_dir = ROOT_DIR / "data/data-splits"
fold_dir = split_dir / "cv_folds_v1"

output_dir = (
    ROOT_DIR / "experiments/v1_baseline/results/evaluation_v1"
)
output_dir.mkdir(parents=True, exist_ok=True)

# Chỉ đọc Development
df_dev = lib.pd.read_csv(
    split_dir / "development.csv",
    encoding="utf-8-sig"
)

assert len(df_dev) == 9000
assert df_dev[id_col].is_unique
assert df_dev[choice_cols].isin([0, 1]).all().all()
assert df_dev[choice_cols].sum(axis=1).eq(4).all()

# Đường dẫn kết quả ba mô hình
model_paths = {
    "Random Forest": ROOT_DIR / (
        "experiments/v1_baseline/random_forest/results/"
        "rf_predictions_5fold.csv"
    ),
    "XGBoost": ROOT_DIR / (
        "experiments/v1_baseline/xgboost/results/"
        "xgb_predictions_5fold.csv"
    ),
    "CatBoost": ROOT_DIR / (
        "experiments/v1_baseline/catboost/results/"
        "cat_predictions_5fold.csv"
    )
}

# Đọc fold đã cố định
fold_ids = {}

for fold in range(1, N_SPLITS + 1):
    train_ids = lib.pd.read_csv(
        fold_dir / f"fold_{fold}_train_ids.csv",
        encoding="utf-8-sig"
    )[id_col]

    val_ids = lib.pd.read_csv(
        fold_dir / f"fold_{fold}_validation_ids.csv",
        encoding="utf-8-sig"
    )[id_col]

    assert set(train_ids).isdisjoint(set(val_ids))

    fold_ids[fold] = {
        "train": train_ids.tolist(),
        "val": val_ids.tolist()
    }

# Đọc kết quả dự đoán
predictions = {}

for model_name, path in model_paths.items():
    pred = lib.pd.read_csv(path, encoding="utf-8-sig")

    assert len(pred) == 9000
    assert pred[id_col].is_unique
    assert pred["Fold"].nunique() == 5
    assert set(pred[id_col]) == set(df_dev[id_col])

    for fold in range(1, N_SPLITS + 1):
        actual_ids = set(
            pred.loc[pred["Fold"] == fold, id_col]
        )
        assert actual_ids == set(fold_ids[fold]["val"])

    # Xác nhận nhãn thực tế không bị lệch ID hoặc thứ tự
    actual_labels = df_dev.set_index(id_col)[choice_cols]
    expected = actual_labels.loc[pred[id_col]].to_numpy()

    true_cols = [f"true_{j}" for j in range(1, 10)]
    pred_cols = [f"pred_{j}" for j in range(1, 10)]

    assert lib.np.array_equal(
        pred[true_cols].to_numpy(), expected
    )
    assert pred[pred_cols].isin([0, 1]).all().all()

    predictions[model_name] = pred

# Tái tạo Most Frequent Combination theo từng fold
baseline_records = []

indexed_dev = df_dev.set_index(id_col)

for fold in range(1, N_SPLITS + 1):
    train_ids = fold_ids[fold]["train"]
    val_ids = fold_ids[fold]["val"]

    y_train = indexed_dev.loc[
        train_ids, choice_cols
    ].to_numpy(dtype=int)

    combos, counts = lib.np.unique(
        y_train,
        axis=0,
        return_counts=True
    )

    most_common = combos[lib.np.argmax(counts)]

    pred_fold = lib.pd.DataFrame({
        id_col: val_ids,
        "Fold": fold
    })

    y_true = indexed_dev.loc[
        val_ids, choice_cols
    ].to_numpy(dtype=int)

    y_pred = lib.np.tile(
        most_common, (len(val_ids), 1)
    )

    for j in range(9):
        pred_fold[f"true_{j+1}"] = y_true[:, j]
        pred_fold[f"pred_{j+1}"] = y_pred[:, j]

    baseline_records.append(pred_fold)

predictions["Most Frequent Combination"] = lib.pd.concat(
    baseline_records,
    ignore_index=True
)

model_order = [
    "Most Frequent Combination",
    "Random Forest",
    "XGBoost",
    "CatBoost"
]

def get_arrays(model_name, fold):
    data = predictions[model_name]
    subset = data[data["Fold"] == fold]

    true_cols = [f"true_{j}" for j in range(1, 10)]
    pred_cols = [f"pred_{j}" for j in range(1, 10)]

    return (
        subset[true_cols].to_numpy(dtype=int),
        subset[pred_cols].to_numpy(dtype=int)
    )

print("PASS: Đã đọc và kiểm tra dự đoán của 4 phương pháp.")
print("Số học sinh Development:", len(df_dev))
print("Số fold:", N_SPLITS)

In [ ]:
# ==========================================
# EXACT-MATCH ACCURACY
# ==========================================

exact_records = []

for model_name in model_order:
    for fold in range(1, N_SPLITS + 1):
        y_true, y_pred = get_arrays(model_name, fold)

        exact_records.append({
            "Model": model_name,
            "Fold": fold,
            "Exact_Match_Accuracy": accuracy_score(
                y_true, y_pred
            )
        })

exact_df = lib.pd.DataFrame(exact_records)

display(
    exact_df.pivot(
        index="Model",
        columns="Fold",
        values="Exact_Match_Accuracy"
    ).round(4)
)

In [ ]:
# ==========================================
# MACRO-F1
# ==========================================

macro_records = []

for model_name in model_order:
    for fold in range(1, N_SPLITS + 1):
        y_true, y_pred = get_arrays(model_name, fold)

        macro_records.append({
            "Model": model_name,
            "Fold": fold,
            "Macro_F1": f1_score(
                y_true, y_pred,
                average="macro",
                zero_division=0
            )
        })

macro_df = lib.pd.DataFrame(macro_records)

display(
    macro_df.pivot(
        index="Model",
        columns="Fold",
        values="Macro_F1"
    ).round(4)
)

In [ ]:
# ==========================================
# MICRO-F1
# ==========================================

micro_records = []

for model_name in model_order:
    for fold in range(1, N_SPLITS + 1):
        y_true, y_pred = get_arrays(model_name, fold)

        micro_records.append({
            "Model": model_name,
            "Fold": fold,
            "Micro_F1": f1_score(
                y_true, y_pred,
                average="micro",
                zero_division=0
            )
        })

micro_df = lib.pd.DataFrame(micro_records)

display(
    micro_df.pivot(
        index="Model",
        columns="Fold",
        values="Micro_F1"
    ).round(4)
)

In [ ]:
# ==========================================
# PRECISION & RECALL
# ==========================================

pr_records = []

for model_name in model_order:
    for fold in range(1, N_SPLITS + 1):
        y_true, y_pred = get_arrays(model_name, fold)

        pr_records.append({
            "Model": model_name,
            "Fold": fold,
            "Macro_Precision": precision_score(
                y_true, y_pred,
                average="macro",
                zero_division=0
            ),
            "Macro_Recall": recall_score(
                y_true, y_pred,
                average="macro",
                zero_division=0
            ),
            "Micro_Precision": precision_score(
                y_true, y_pred,
                average="micro",
                zero_division=0
            ),
            "Micro_Recall": recall_score(
                y_true, y_pred,
                average="micro",
                zero_division=0
            )
        })

pr_df = lib.pd.DataFrame(pr_records)

display(
    pr_df.groupby("Model")[
        [
            "Macro_Precision",
            "Macro_Recall",
            "Micro_Precision",
            "Micro_Recall"
        ]
    ].mean().round(4)
)

In [ ]:
# ==========================================
# PER-LABEL F1-SCORE
# ==========================================

label_records = []

for model_name in model_order:
    for fold in range(1, N_SPLITS + 1):
        y_true, y_pred = get_arrays(model_name, fold)

        scores = f1_score(
            y_true, y_pred,
            average=None,
            zero_division=0
        )

        for label, score in zip(choice_cols, scores):
            label_records.append({
                "Model": model_name,
                "Fold": fold,
                "Nhãn": label,
                "F1": score
            })

label_f1_df = lib.pd.DataFrame(label_records)

label_f1_summary = (
    label_f1_df.groupby(["Nhãn", "Model"])["F1"]
    .mean()
    .unstack("Model")
    .reindex(columns=model_order)
)

display(label_f1_summary.round(4))

In [ ]:
# ==========================================
# HAMMING LOSS
# ==========================================

hamming_records = []

for model_name in model_order:
    for fold in range(1, N_SPLITS + 1):
        y_true, y_pred = get_arrays(model_name, fold)

        hamming_records.append({
            "Model": model_name,
            "Fold": fold,
            "Hamming_Loss": hamming_loss(
                y_true, y_pred
            )
        })

hamming_df = lib.pd.DataFrame(hamming_records)

display(
    hamming_df.pivot(
        index="Model",
        columns="Fold",
        values="Hamming_Loss"
    ).round(4)
)

In [ ]:
# ==========================================
# CONFUSION MATRIX - 9 NHÃN / MÔ HÌNH
# ==========================================

cm_dir = output_dir / "confusion_matrices"
cm_dir.mkdir(parents=True, exist_ok=True)

cm_records = []

for model_name in model_order:

    data = predictions[model_name]

    true_cols = [f"true_{j}" for j in range(1, 10)]
    pred_cols = [f"pred_{j}" for j in range(1, 10)]

    y_true = data[true_cols].to_numpy(dtype=int)
    y_pred = data[pred_cols].to_numpy(dtype=int)

    fig, axes = lib.plt.subplots(
        3, 3, figsize=(15, 13)
    )

    for j, label in enumerate(choice_cols):

        cm = confusion_matrix(
            y_true[:, j],
            y_pred[:, j],
            labels=[0, 1]
        )

        tn, fp, fn, tp = cm.ravel()

        cm_records.append({
            "Model": model_name,
            "Nhãn": label,
            "TN": int(tn),
            "FP": int(fp),
            "FN": int(fn),
            "TP": int(tp)
        })

        ax = axes.flat[j]

        ax.imshow(cm, cmap="Blues")

        for row in range(2):
            for col in range(2):
                ax.text(
                    col, row,
                    f"{cm[row, col]:,}",
                    ha="center",
                    va="center",
                    color=(
                        "white"
                        if cm[row, col] > cm.max() / 2
                        else "black"
                    )
                )

        ax.set_title(label, fontsize=10)
        ax.set_xticks([0, 1])
        ax.set_yticks([0, 1])
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")

    fig.suptitle(
        f"Confusion Matrices - {model_name}",
        fontsize=15
    )

    fig.tight_layout()

    filename = (
        model_name.lower()
        .replace(" ", "_")
        + "_confusion_matrices.png"
    )

    fig.savefig(
        cm_dir / filename,
        dpi=200,
        bbox_inches="tight"
    )

    lib.plt.show()
    lib.plt.close(fig)

cm_df = lib.pd.DataFrame(cm_records)

display(cm_df)

In [ ]:
# ==========================================
# KIỂM TRA RÀNG BUỘC TOP-4
# ==========================================

top4_records = []

for model_name in model_order:

    data = predictions[model_name]
    pred_cols = [f"pred_{j}" for j in range(1, 10)]

    y_pred = data[pred_cols].to_numpy(dtype=int)

    valid = (y_pred.sum(axis=1) == 4)

    top4_records.append({
        "Model": model_name,
        "Tổng mẫu": len(y_pred),
        "Số mẫu đúng 4 nhãn": int(valid.sum()),
        "Số mẫu vi phạm": int((~valid).sum()),
        "Tỷ lệ hợp lệ (%)": round(valid.mean() * 100, 2)
    })

top4_df = lib.pd.DataFrame(top4_records)

display(top4_df)

assert (top4_df["Số mẫu vi phạm"] == 0).all()

print("PASS: Tất cả dự đoán đều chọn đúng 4/9 môn.")

In [ ]:
# ==========================================
# KIỂM TRA TỔ HỢP DỰ ĐOÁN HỢP LỆ
# ==========================================

# Các tổ hợp đã quan sát trong Development
observed_combos = set(
    map(
        tuple,
        df_dev[choice_cols].to_numpy(dtype=int)
    )
)

assert len(observed_combos) == 18

combination_records = []

for model_name in model_order:

    data = predictions[model_name]
    pred_cols = [f"pred_{j}" for j in range(1, 10)]

    y_pred = data[pred_cols].to_numpy(dtype=int)

    predicted_combos = list(map(tuple, y_pred))

    valid = lib.np.array([
        combo in observed_combos
        for combo in predicted_combos
    ])

    combination_records.append({
        "Model": model_name,
        "Tổng mẫu": len(y_pred),
        "Số tổ hợp dự đoán khác nhau": len(
            set(predicted_combos)
        ),
        "Số mẫu thuộc 18 tổ hợp": int(valid.sum()),
        "Số mẫu ngoài 18 tổ hợp": int((~valid).sum()),
        "Tỷ lệ thuộc 18 tổ hợp (%)": round(
            valid.mean() * 100, 2
        )
    })

combination_df = lib.pd.DataFrame(combination_records)

display(combination_df)

In [ ]:
# ==========================================
# TỔNG HỢP METRICS MEAN ± STD
# ==========================================

metrics_5fold = exact_df.copy()

for metric_df in [
    macro_df,
    micro_df,
    pr_df,
    hamming_df
]:
    metrics_5fold = metrics_5fold.merge(
        metric_df,
        on=["Model", "Fold"],
        validate="one_to_one"
    )

metric_cols = [
    "Exact_Match_Accuracy",
    "Macro_F1",
    "Micro_F1",
    "Macro_Precision",
    "Macro_Recall",
    "Micro_Precision",
    "Micro_Recall",
    "Hamming_Loss"
]

summary_records = []

for model_name in model_order:

    subset = metrics_5fold[
        metrics_5fold["Model"] == model_name
    ]

    for metric in metric_cols:
        summary_records.append({
            "Model": model_name,
            "Metric": metric,
            "Mean": subset[metric].mean(),
            "Std": subset[metric].std(ddof=1)
        })

evaluation_summary = lib.pd.DataFrame(summary_records)

evaluation_summary["Mean ± Std"] = evaluation_summary.apply(
    lambda row: f"{row['Mean']:.4f} ± {row['Std']:.4f}",
    axis=1
)

display(
    evaluation_summary.pivot(
        index="Metric",
        columns="Model",
        values="Mean ± Std"
    ).reindex(columns=model_order)
)

# Kiểm tra đúng 5 fold cho mỗi mô hình
assert (
    metrics_5fold.groupby("Model")["Fold"].nunique() == 5
).all()

In [ ]:
# ==========================================
# SO SÁNH 4 PHƯƠNG PHÁP
# ==========================================

comparison_df = evaluation_summary.pivot(
    index="Model",
    columns="Metric",
    values="Mean"
).reindex(model_order)

# Lấy Reference làm mốc
reference = comparison_df.loc[
    "Most Frequent Combination"
]

comparison_df["Delta_Exact_vs_Reference"] = (
    comparison_df["Exact_Match_Accuracy"]
    - reference["Exact_Match_Accuracy"]
)

comparison_df["Delta_MacroF1_vs_Reference"] = (
    comparison_df["Macro_F1"]
    - reference["Macro_F1"]
)

comparison_df["Delta_MicroF1_vs_Reference"] = (
    comparison_df["Micro_F1"]
    - reference["Micro_F1"]
)

comparison_df["Delta_Hamming_vs_Reference"] = (
    comparison_df["Hamming_Loss"]
    - reference["Hamming_Loss"]
)

print("SO SÁNH V1 VỚI BASELINE REFERENCE")
display(comparison_df.round(4))

# Biểu đồ Exact-match Accuracy
accuracy_values = (
    comparison_df["Exact_Match_Accuracy"] * 100
)

fig, ax = lib.plt.subplots(figsize=(10, 5))

bars = ax.bar(
    accuracy_values.index,
    accuracy_values.values
)

ax.set_title("V1 - Exact-match Accuracy")
ax.set_ylabel("Accuracy (%)")
ax.set_ylim(0, max(accuracy_values) * 1.2)

ax.bar_label(bars, fmt="%.2f%%", padding=3)

lib.plt.xticks(rotation=15)
lib.plt.tight_layout()

fig.savefig(
    output_dir / "comparison_exact_match.png",
    dpi=200,
    bbox_inches="tight"
)

lib.plt.show()

In [ ]:
# ==========================================
# XÁC ĐỊNH BEST MODEL V1
# LƯU BÁO CÁO ĐÁNH GIÁ
# ==========================================

import json

# Chỉ xét ba thuật toán Machine Learning
ml_models = [
    "Random Forest",
    "XGBoost",
    "CatBoost"
]

ml_comparison = comparison_df.loc[ml_models].copy()

ranking = ml_comparison.sort_values(
    by=[
        "Exact_Match_Accuracy",
        "Macro_F1",
        "Micro_F1",
        "Hamming_Loss"
    ],
    ascending=[False, False, False, True]
)

best_ml_model = ranking.index[0]

# So sánh với phương pháp tham chiếu
overall_ranking = comparison_df.sort_values(
    by=[
        "Exact_Match_Accuracy",
        "Macro_F1",
        "Micro_F1",
        "Hamming_Loss"
    ],
    ascending=[False, False, False, True]
)

best_overall = overall_ranking.index[0]

print("=" * 60)
print("MODEL EVALUATION V1 - KẾT LUẬN")
print("=" * 60)

print("Mô hình ML tốt nhất:", best_ml_model)
print(
    "Exact-match Accuracy:",
    f"{ranking.iloc[0]['Exact_Match_Accuracy']:.4f}"
)

print("Phương pháp tốt nhất toàn bộ:", best_overall)

print("\nXẾP HẠNG MACHINE LEARNING")
display(ranking.round(4))

# Lưu các báo cáo CSV
reports = {
    "metrics_5fold.csv": metrics_5fold,
    "evaluation_summary.csv": evaluation_summary,
    "label_f1_summary.csv": label_f1_summary.reset_index(),
    "confusion_matrix_summary.csv": cm_df,
    "top4_validation.csv": top4_df,
    "combination_validation.csv": combination_df,
    "model_comparison.csv": comparison_df.reset_index(),
    "ml_model_ranking.csv": ranking.reset_index()
}

for filename, data in reports.items():
    data.to_csv(
        output_dir / filename,
        index=False,
        encoding="utf-8-sig"
    )

# Lưu kết luận
conclusion = {
    "experiment": "V1 - Baseline",
    "random_seed": RANDOM_SEED,
    "n_splits": N_SPLITS,
    "n_development": 9000,
    "primary_metric": "Exact_Match_Accuracy",
    "best_ml_model": best_ml_model,
    "best_ml_exact_match": float(
        ranking.iloc[0]["Exact_Match_Accuracy"]
    ),
    "best_overall_method": best_overall,
    "best_overall_exact_match": float(
        overall_ranking.iloc[0]["Exact_Match_Accuracy"]
    ),
    "independent_test_used": False
}

with open(
    output_dir / "v1_conclusion.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        conclusion,
        f,
        ensure_ascii=False,
        indent=4
    )

print("\nPASS: Đã lưu toàn bộ báo cáo Evaluation V1.")
print("Thư mục:", output_dir.resolve())